In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# gpt-4o-mini 는 멀티모달 — 이미지 입력을 그대로 받는다
llm = init_chat_model("openai:gpt-4o-mini", temperature=0.7)

In [ ]:
import base64
import mimetypes


def load_image(path: str) -> str:
    """이미지 파일 → data URL (base64). 비전 모델 입력용."""
    mime = mimetypes.guess_type(path)[0] or "image/jpeg"
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    return f"data:{mime};base64,{b64}"

In [ ]:
import wikipedia
from langchain_core.tools import tool


@tool
def search_wikipedia(query: str) -> str:
    """미술 작가·작품의 사실 정보(생몰년·제작 연도·배경 등)를 위키피디아에서 조회한다.
    정확도를 위해 영문 작가명 또는 작품명으로 검색하라. 예: 'Vincent van Gogh', 'The Starry Night'."""
    wikipedia.set_lang("en")
    try:
        return wikipedia.summary(query, sentences=6, auto_suggest=True)
    except wikipedia.DisambiguationError as e:
        return wikipedia.summary(e.options[0], sentences=6)
    except wikipedia.PageError:
        hits = wikipedia.search(query)
        return wikipedia.summary(hits[0], sentences=6) if hits else "관련 문서를 찾지 못했습니다."
    except Exception as ex:
        return f"조회 실패: {ex}"


# 14.5 에서 배운 bind_tools — 모델에게 툴 사용법을 알려준다
llm_with_tools = llm.bind_tools([search_wikipedia])

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph.message import add_messages


class DocentState(TypedDict):
    messages: Annotated[list, add_messages]
    persona: Literal["child", "beginner", "explorer", "healing"]  # 관람객 유형 = 학습 수준
    image: str             # 입력: 이미지 (base64 data URL)
    image_info: str        # router가 채움: 트랙 A일 때 식별된 작가·작품명
    track: Literal["A", "C"]  # router 결과: A=명화 / C=미상
    art_context: str       # analyze 출력: 해설 재료
    docent_script: str     # generate_docent 출력


PERSONA_TONE = {
    "child":    "8살 어린이에게 말하듯. 쉬운 비유와 질문을 섞어 짧고 재미있게. 어려운 용어 금지.",
    "beginner": "미술 입문자에게. 일상적인 비유로 공감대를 만들고 쉬운 단어 중심으로.",
    "explorer": "지식 탐구자에게. 전문 용어와 미술사적 맥락·시대상을 담아 평론 스타일로.",
    "healing":  "위로를 원하는 사람에게. 에세이 톤으로 차분하고 감성적인 묘사 중심으로.",
}

In [ ]:
from langchain_core.messages import HumanMessage


def router(state: DocentState) -> dict:
    """이미지를 직접 보고 유명 명화(A)인지 정보 없는 이미지(C)인지 판단 + 작품 식별."""
    resp = llm.invoke([
        HumanMessage(content=[
            {
                "type": "text",
                "text": """이 이미지가 미술사에 기록된 '유명 명화'인지 판단해줘.

                작가·작품이 확실히 특정되는 유명 명화면  ->  A | 작가, 작품명
                그 외 (AI 생성물·일반 이미지·확신 없음)  ->  C

                위 형식으로 한 줄만 답해.""",
            },
            {"type": "image_url", "image_url": {"url": state["image"]}},
        ])
    ])
    text = resp.content.strip()

    if text.upper().startswith("A"):
        identity = text.split("|", 1)[1].strip() if "|" in text else ""
        print(f"[router] track = A  ({identity})")
        return {"track": "A", "image_info": identity}

    print("[router] track = C")
    return {"track": "C"}


def route_by_track(state: DocentState) -> Literal["analyze_masterpiece", "analyze_mood"]:
    return "analyze_masterpiece" if state["track"] == "A" else "analyze_mood"

In [ ]:
ART_INSTRUCTION = """너는 미술사 전문가야. 관람객에게 들려줄 도슨트 해설의 '재료'를 준비해.
작가 생몰·시대, 제작 연도, 배경/일화 같은 사실은 반드시 search_wikipedia 툴로 조회해 근거를 확보해.
정보가 충분하면 툴을 더 부르지 말고, 아래 형식으로 정리해:
1. 작가와 시대  2. 배경/비하인드  3. 화풍의 특징(색채·구도·기법)"""


def analyze_masterpiece(state: DocentState) -> dict:
    """트랙 A: Wikipedia 툴로 사실을 조회하며 명화 맥락을 정리 (tool loop)."""
    msgs = state["messages"]
    seed = []
    if not msgs:  # 첫 진입: 지시 + 식별된 작품명을 대화에 심는다
        seed = [HumanMessage(content=f"{ART_INSTRUCTION}\n\n작품: {state['image_info']}")]
        msgs = seed

    response = llm_with_tools.invoke(msgs)
    out = {"messages": seed + [response]}
    if not response.tool_calls:      # 툴 호출이 없으면 = 정리 완료
        out["art_context"] = response.content
    return out


def analyze_mood(state: DocentState) -> dict:
    """트랙 C: 이미지를 비전으로 분석 → 색감·질감·무드."""
    resp = llm.invoke([
        HumanMessage(content=[
            {
                "type": "text",
                "text": """너는 비주얼 아트 큐레이터야.
                이 이미지의 무드를 분석해줘.

                1. 지배적인 색감·톤
                2. 질감·구도의 분위기
                3. 감정 키워드 3개""",
            },
            {"type": "image_url", "image_url": {"url": state["image"]}},
        ])
    ])
    return {"art_context": resp.content, "messages": [resp]}

In [ ]:
def generate_docent(state: DocentState) -> dict:
    """맥락 + 페르소나 → 눈높이 맞춤 도슨트 해설."""
    resp = llm.invoke(
        f"""너는 미술관 도슨트야.
        아래 작품 맥락으로 관람객에게 들려줄 해설을 써줘.

        [작품 맥락]
        {state['art_context']}

        [관람객 눈높이]
        {PERSONA_TONE[state['persona']]}

        위 눈높이에 맞는 톤으로 3~4문단 해설을 작성해줘."""
    )
    return {"docent_script": resp.content, "messages": [resp]}

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

builder = StateGraph(DocentState)
builder.add_node("router", router)
builder.add_node("analyze_masterpiece", analyze_masterpiece)
builder.add_node("tools", ToolNode([search_wikipedia]))   # Wikipedia 툴 실행 노드
builder.add_node("analyze_mood", analyze_mood)
builder.add_node("generate_docent", generate_docent)

builder.add_edge(START, "router")
builder.add_conditional_edges("router", route_by_track)   # ① 트랙 A/C 분기

# ② tool loop: 툴 호출 있으면 tools 로, 없으면 generate_docent 로
builder.add_conditional_edges(
    "analyze_masterpiece", tools_condition,
    {"tools": "tools", END: "generate_docent"},
)
builder.add_edge("tools", "analyze_masterpiece")          # 툴 결과 들고 다시 정리

builder.add_edge("analyze_mood", "generate_docent")
builder.add_edge("generate_docent", END)

graph = builder.compile()

In [ ]:
print(graph.get_graph().draw_ascii())

In [ ]:
def run_docent(image_path: str, persona: str):
    result = graph.invoke({
        "image": load_image(image_path),
        "image_info": "",
        "persona": persona,
        "messages": [],
    })
    print(f"===== {image_path} · track {result['track']} · persona {persona} =====")
    if result["track"] == "A":
        print(f"(식별: {result['image_info']})")
    print(result["docent_script"], "\n")
    return result

In [ ]:
STARRY = "The Starry Night.jpg"   # 실제 명화 → 트랙 A → Wikipedia 툴로 근거 조회

run_docent(STARRY, "child")
run_docent(STARRY, "explorer")

In [ ]:
run_docent("anyimage.png", "healing")   # AI 생성 이미지 → 트랙 C (툴 미사용, 비전 분석)